# Feature Engineering: Categorical Encoding



## Setup & Synthetic Data Generation

Let's create a synthetic dataset with various types of categorical variables to demonstrate these encodings.

In [10]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# We will need category_encoders for advanced techniques
# !pip install category_encoders 
try:
    import category_encoders as ce
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "category_encoders"])
    import category_encoders as ce

np.random.seed(42)

# Creating a synthetic dataset with diverse types of categorical data
data = {
    'Size': ['Small', 'Medium', 'Large', 'Medium', 'Small', 'Large', 'Medium', 'Small', 'Large', 'Medium'], # Ordinal
    'Color': ['Red', 'Blue', 'Green', 'Red', 'Red', 'Blue', 'Green', 'Blue', 'Green', 'Red'], # Nominal (Low Cardinality)
    'City': ['NYC', 'LA', 'SF', 'NYC', 'Chicago', 'LA', 'Seattle', 'Boston', 'NYC', 'SF'], # Nominal (High Cardinality)
    'Target': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0] # Binary Target for classification tasks
}

df_original = pd.DataFrame(data)
display(df_original.head())


,Size,Color,City,Target
0,Small,Red,NYC,1
1,Medium,Blue,LA,0
2,Large,Green,SF,1
3,Medium,Red,NYC,0
4,Small,Red,Chicago,1


## 1. Label Encoding (Simple)
**What it is:** Assigns a random or alphabetical unique integer to each category (e.g., Red=0, Blue=1, Green=2).

**Why to use it:** Highly memory efficient. It keeps the feature space the same (1 column) and converts strings to optimized numeric types.

**When TO use it:** 
- Use to encode your **Target Variable** (`y`) for classification tasks. Scikit-Learn strictly requires numeric target labels.
- Can be used for **Tree-based models** (Random Forest, XGBoost) because trees can handle arbitrary splits, however Ordinal or Target encoding is often better.

**When NOT to use it:**
- **DO NOT USE** for linear models (Linear Regression, Logistic, SVM, Neural Nets). 
- **Reason:** It introduces a false mathematical relationship. The model will think `Green (2)` is "greater than" `Red (0)`, which is factually incorrect for colors and will ruin the model's math.

**How to use (step-by-step):**
1. Fit the encoder **only on the training labels**.
2. Save the label-to-number mapping for validation/test sets.
3. Use `inverse_transform()` to convert predictions back to original labels for reporting.

**Example Scenario:**
Encoding the target variable column "Outcome" where "Failed" becomes 0, "Passed" becomes 1, and "Pending" becomes 2.


In [11]:
from sklearn.preprocessing import LabelEncoder

df = df_original.copy()
le = LabelEncoder()
df['Color_Label_Encoded'] = le.fit_transform(df['Color'])

print("Label Encoding Example:")
display(df[['Color', 'Color_Label_Encoded']].head())


Label Encoding Example:


,Color,Color_Label_Encoded
0,Red,2
1,Blue,0
2,Green,1
3,Red,2
4,Red,2


## 2. Ordinal Encoding (Simple)
**What it is:** Similar to Label Encoding, but the integers are explicitly assigned based on a predefined **logical order** you provide (e.g., Small=0, Medium=1, Large=2).

**Why to use it:** It mathematically preserves the natural ranking relationship present within the data categories.

**When TO use it:** 
- Always use when the categorical variable has a **natural, meaningful order**.

**When NOT to use it:**
- Do not use for nominal (unordered) categories. For example, assigning integers to "Countries". Is France higher than Germany? No.

**How to use (step-by-step):**
1. Write down the true order of categories (business logic > alphabetical order).
2. Map categories to increasing integers using a dictionary.
3. Keep the mapping consistent across train/validation/test sets.

**Example Scenario:**
Encoding "Education Level" where High School = 0, Bachelors = 1, Masters = 2, PhD = 3. Now the model correctly understands that PhD is geometrically higher/greater than High School.


In [12]:
df = df_original.copy()

# Define the logical order explicitly using a dictionary
size_mapping = {'Small': 0, 'Medium': 1, 'Large': 2}
df['Size_Ordinal_Encoded'] = df['Size'].map(size_mapping)

print("Ordinal Encoding Example:")
display(df[['Size', 'Size_Ordinal_Encoded']].head())


Ordinal Encoding Example:


,Size,Size_Ordinal_Encoded
0,Small,0
1,Medium,1
2,Large,2
3,Medium,1
4,Small,0


## 3. One-Hot Encoding (OHE) / Dummy Encoding (Simple/Standard)
**What it is:** Creates a brand new binary (1/0) column for *every single distinct category*. E.g., `is_Red` (1/0), `is_Blue` (1/0). 
*(Note: Dummy encoding is the same as OHE, but it drops one column to avoid statistical multicollinearity).*

**Why to use it:** It does not assume any ordering or mathematical relationship between categories whatsoever. It explicitly isolates every category's independent effect.

**When TO use it:** 
- Best for categorical variables with **low cardinality** (few unique lists, generally < 15 options). 
- Highly recommended and often practically necessary for **Linear Regression, Logistic Regression, and Neural Networks**.

**When NOT to use it:**
- **DO NOT USE** for high cardinality variables (like Zip Codes, City Names). 
- **Reason:** If you have 5,000 zip codes, OHE will create 5,000 new columns! This causes the "Curse of Dimensionality", drastically slows down computing, blows up RAM usage, and makes Tree models hopelessly ineffective because the data becomes extremely sparse (filled with zeros).

**How to use (step-by-step):**
1. Identify low-cardinality categorical columns.
2. Use `pd.get_dummies()` or `OneHotEncoder` from scikit-learn.
3. Store the generated column list to ensure test data aligns with training.
4. Consider `drop_first=True` for linear models to avoid multicollinearity.

**Example Scenario:**
Encoding a "Gender" column (Male/Female/Other) into 3 separate `is_Male`, `is_Female`, `is_Other` columns.


In [13]:
df = df_original.copy()

# Using Pandas get_dummies 
# drop_first=False is Standard OHE
df_ohe = pd.get_dummies(df, columns=['Color'], prefix='Color') 

# drop_first=True is Dummy Encoding (removes Red to avoid multicollinearity)
df_dummy = pd.get_dummies(df, columns=['Color'], prefix='Color', drop_first=True) 

print("One-Hot Encoding (Pandas):")
display(df_ohe.head())

# One-Hot Encoding using scikit-learn
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
color_ohe = ohe.fit_transform(df[['Color']])
ohe_df = pd.DataFrame(color_ohe, columns=ohe.get_feature_names_out(['Color']))

print("One-Hot Encoding (scikit-learn):")
display(ohe_df.head())


One-Hot Encoding (Pandas):


,Size,City,Target,Color_Blue,Color_Green,Color_Red
0,Small,NYC,1,False,False,True
1,Medium,LA,0,True,False,False
2,Large,SF,1,False,True,False
3,Medium,NYC,0,False,False,True
4,Small,Chicago,1,False,False,True


One-Hot Encoding (scikit-learn):


,Color_Blue,Color_Green,Color_Red
0,0.0,0.0,1.0
1,1.0,0.0,0.0
2,0.0,1.0,0.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0


## 4. Frequency / Count Encoding (Intermediate)
**What it is:** Replaces the text category entirely with the number of times it appears in the dataset. (e.g., if "NYC" appears 300 times in the data, replace "NYC" with the number 300).

**Why to use it:** Helps the model understand how common or rare a category is without expanding the feature space (it stays 1 column). 

**When TO use it:** 
- Good for **tree-based models**. 
- Useful for **high cardinality** categorical features where OHE is impossible. 
- Useful when you strongly suspect the frequency of a category correlates with the target.

**When NOT to use it:**
- Do not use if two entirely different categories happen to have the exact same frequency count. They will both be assigned the identical number, wiping out the model's ability to distinguish between them! (This is called a "collision").

**How to use (step-by-step):**
1. Compute value counts on the **training data** only.
2. Map each category to its count (or proportion).
3. For unseen categories in test data, assign 0 or the global minimum count.

**Example Scenario:**
In fraud detection, rare IP addresses or rare browsers might be more suspicious. Replacing the IP address string with its "count" helps the model instantly see "Oh, this IP has a count of 1, it's very rare."


In [14]:
df = df_original.copy()

# Count Encoding mapping
count_map = df['City'].value_counts().to_dict()
df['City_Count_Encoded'] = df['City'].map(count_map)

print("Count Encoding Example (Number of times the city appears in dataset):")
display(df[['City', 'City_Count_Encoded']].head())


Count Encoding Example (Number of times the city appears in dataset):


,City,City_Count_Encoded
0,NYC,3
1,LA,2
2,SF,2
3,NYC,3
4,Chicago,1


## 5. Target Encoding (Mean Encoding) [Advanced]
**What it is:** Replaces a category with the *average of the target variable* for that specific category. (e.g., If the average house price in NYC is $800k, then "NYC" is replaced by the number 800000).

**Why to use it:** Directly captures the relationship between the category and the target outcome, providing a massively strong, predictive signal to the model.

**When TO use it:** 
- The absolute best choice for **high-cardinality** features (e.g., zip codes, cities, product IDs) in **tree-based models** (XGBoost, LightGBM).

**When NOT to use it:**
- If you have very few samples (e.g., only 1 row for "Boston"). If "Boston" happens to have a Target of 1, the model learns that Boston *always* equals 1. This causes massive **Target Leakage and Overfitting**.
- **Rule:** You MUST use "smoothing" (which pulls averages of rare categories toward the global average) and you MUST fit it inside your cross-validation loops, never on the whole dataset at front.

**How to use (step-by-step):**
1. Split data into train/validation (or use cross-validation).
2. Fit the encoder **only** on the training folds.
3. Apply smoothing/regularization to reduce leakage on rare categories.
4. Transform validation/test data using the fitted encoder.

**Example Scenario:**
You are predicting if someone will default on a loan (Target 1 or 0). You replace the "Zip Code" column with the historical *default rate* of that Zip Code (e.g., Zip Code 90210 gets replaced by the number 0.04 (4% default rate)).


In [15]:
df = df_original.copy()

# We must fit TargetEncoder only on training data to avoid severe target leakage!
# The smoothing parameter pulls rare categories (like a City that only appeared once) towards the global mean target
target_enc = ce.TargetEncoder(cols=['City'], smoothing=10)
df['City_Target_Encoded'] = target_enc.fit_transform(df['City'], df['Target'])

print("Target Encoding Example:")
display(df[['City', 'Target', 'City_Target_Encoded']].sort_values('City').head(10))


Target Encoding Example:


,City,Target,City_Target_Encoded
7,Boston,0,0.434946
4,Chicago,1,0.565054
1,LA,0,0.429074
5,LA,0,0.429074
0,NYC,1,0.525744
3,NYC,0,0.525744
8,NYC,1,0.525744
2,SF,1,0.500000
9,SF,0,0.500000
6,Seattle,1,0.565054


## 6. Leave-One-Out Encoding (LOOE) [Advanced]
**What it is:** A strict upgrade to Target Encoding. Imagine you are trying to guess the average test score for Class A, and you are in Class A. If you include *your own* 100% test score, you artificially inflate the average you see! 
LOOE fixes this: For your row, it replaces your category with the average target of *everyone else* in Class A, explicitly leaving your row's target out of the math.

**Why to use it:** It structurally prevents Target Leakage. By not letting a row see its own target value during the encoding process, the model is forced to generalize better.

**When TO use it:** 
- Used in the exact same scenarios as Target Encoding (High cardinality, Tree models). It is the preferred, safer alternative when standard Target Encoding is overfitting your training data.

**When NOT to use it:**
- Do not use for small datasets or inside linear models where standard Target Encoding with heavily regularized smoothing might be more computationally efficient.

**How to use (step-by-step):**
1. Fit the Leave-One-Out encoder on training data with the target.
2. Apply it within cross-validation to avoid leakage.
3. Consider adding noise (regularization) if the model still overfits.

**Example Scenario:**
Row 1 is "NYC" with Target 1. Row 2 is "NYC" with Target 0. Row 3 is "NYC" with Target 1.
For Row 1's encoding, it averages Row 2 and 3: (0 + 1) / 2 = 0.5. NYC for Row 1 is encoded as 0.5.


In [16]:
df = df_original.copy()

loo_enc = ce.LeaveOneOutEncoder(cols=['City'])
df['City_LOO_Encoded'] = loo_enc.fit_transform(df['City'], df['Target'])

print("Leave-One-Out Encoding Example:")
display(df[['City', 'Target', 'City_LOO_Encoded']].sort_values('City').head(10))


Leave-One-Out Encoding Example:


,City,Target,City_LOO_Encoded
7,Boston,0,0.5
4,Chicago,1,0.5
1,LA,0,0.0
5,LA,0,0.0
0,NYC,1,0.5
3,NYC,0,1.0
8,NYC,1,0.5
2,SF,1,0.0
9,SF,0,1.0
6,Seattle,1,0.5


## 7. Hashing Encoding (Feature Hashing / Hashing Trick) [Advanced]
**What it is:** Imagine a physical dictionary where you look up a city name to get its numeric code. For millions of cities, that dictionary takes up massive RAM. 
A Hashing function is like a mathematical blender: you throw the string "Chicago" into it, and it reliably spits out the number `3` every single time. It doesn't need to remember or store the dictionary! It just maps the text to a fixed set of columns.

**Why to use it:** Super fast and highly memory efficient. It handles completely unseen categories (that weren't in the training set) automatically on the fly.

**When TO use it:** 
- Mandatory for **truly massive datasets** with extreme cardinality (e.g. Millions of User IDs, IP Addresses, URLs).
- Used extensively in **streaming learning / online learning** where data comes in continuously and the vocabulary is infinite and unknown.

**When NOT to use it:**
- Do not use for normal or small datasets! 
- **The Drawback (Hash Collisions):** Since the mathematical blender only has so many outputs (e.g. 100 columns), two entirely different strings (like "Boston" and "Malware_IP_Address") might accidentally get mathematically blended into the exact same column `3`. The model can no longer tell them apart!

**How to use (step-by-step):**
1. Choose `n_components` (number of hash bins) based on dataset size and memory limits.
2. Fit/transform training data; apply the same encoder to validation/test data.
3. Monitor collision risk by trying a few `n_components` values and checking model quality.

**Example Scenario:**
You are Google, building an ad-click predictor. A new user with a totally unique `Device_ID` connects. One-Hot encoding would crash, and Target Encoding has never seen this ID before. Hashing Encoding simply hashes the ID text string into column `74` instantly and keeps processing.


In [17]:
df = df_original.copy()

# We force the math blender to only output into 4 predefined columns (n_components=4)
# No matter if we have 10 cities or 10 Million cities, it will only create 4 columns.
hash_enc = ce.HashingEncoder(cols=['City'], n_components=4)
df_hash = hash_enc.fit_transform(df)

print("Hashing Encoding Example:")
display(df_hash[['col_0', 'col_1', 'col_2', 'col_3']].head())


Hashing Encoding Example:


,col_0,col_1,col_2,col_3
0,0,1,0,0
1,0,1,0,0
2,0,1,0,0
3,0,1,0,0
4,0,0,1,0


## 8. Special Strategy: Handling Very High Cardinality (Zip Codes, Cities, IDs)
**The Problem:** When a feature has hundreds or thousands of unique categories (like zip codes, product IDs, detailed city names), standard encoding techniques fail.
- **One-Hot Encoding** creates thousands of columns, destroying model performance, causing out-of-memory errors, and making trees hopelessly sparse (the curse of dimensionality).
- **Label/Ordinal Encoding** sets arbitrary integer assignments that trees will struggle to split logically without building massive depth.

**How to handle it (From best to worst):**
1. **Target Encoding / Leave-One-Out Encoding (LOOE):** This is usually the **gold standard** for machine learning models (especially Tree-Based). It collapses thousands of categories into a single highly predictive numeric column. You *must* use smoothing/regularization to prevent overfitting on rare zip codes (e.g. a zip code with only 1 sample).
2. **Frequency / Count Encoding:** A safe, simple baseline. If a zip code is common, it gets a high count; if rare, a low count. Works surprisingly well if the "popularity" of a category correlates with the target.
3. **Hashing Encoding:** Use this if the cardinality is so huge (e.g. millions of IP addresses/URLs) that your RAM blows up just trying to maintain a Target Encoding dictionary. Hashing maps it to a fixed number of dimensions automatically.
4. **Entity Embeddings (Deep Learning):** If using Neural Networks, passing the high cardinality feature through an Embedding layer (like word embeddings in NLP) is the absolute best approach. It allows the network to learn rich, multi-dimensional representations of categorical levels, clustering similar zip codes together in a continuous vector space.




## 🌟 Cheat Sheet: What to use and When?

### 1. By Cardinality (Number of Unique Categories):
- **Very Low Cardinality (2-15):** One-Hot Encoding / Dummy Encoding.
- **Moderate to High Cardinality (15 - 500):** Target Encoding, Leave-One-Out Encoding, Frequency Encoding.
- **Extremely High Cardinality (10,000+):** Hashing Encoding, Target Encoding.

### 2. By Model Type:
- **Linear Models (Linear/Logistic Regression, SVM):** One-Hot Encoding. Avoid Label Encoding (unless variables are strictly ordinal).
- **Tree-Based Models (Random Forest, XGBoost, LightGBM):** Ordinal Encoding, Target Encoding, Frequency Encoding. Tree models can split on ordinal integers effectively to group categories, so OHE is often unnecessary and can make trees slow, bloated, and sparse. *(Note: LightGBM and CatBoost have built-in categorical handling which is often highly optimal!).*
- **Neural Networks:** One-Hot Encoding, Target Encoding, or Entity Embeddings (advanced DL technique).

### 3. By Data Relationship:
- **Has an inherent logical order:** Ordinal Encoding.
- **No inherent order:** Any of the nominal encoding techniques (OHE, Target, Hashing, etc.).
